# Ticket 1 - Inventario y validacion documental en Colab

Este notebook funciona como laboratorio y evidencia del **Ticket 1** del proyecto **BlueSea AI Assistant**.

Objetivo del notebook:

- instalar el proyecto en Google Colab;
- revisar la estructura documental definida;
- validar el inventario maestro de 24 documentos;
- generar el reporte `data/processed/document_status.csv`;
- generar el inventario tecnico `data/processed/inventory.json`;
- identificar documentos faltantes sin modificar el CSV oficial.

Este cuaderno no procesa chunks ni embeddings. Eso corresponde al Ticket 2.

## 1. Clonar el repositorio

Ejecuta esta celda en Google Colab. Si el repositorio ya existe en la sesion, puedes reiniciar el runtime o borrar la carpeta antes de clonar otra vez.

In [ ]:
!git clone https://github.com/aantoa/bluesea-ai-assistant.git
%cd bluesea-ai-assistant

## 2. Instalar dependencias y paquete local

`requirements.txt` incluye `-e .`, por eso Colab instala el paquete desde `src/rag_bsf/` y permite ejecutar `python -m rag_bsf.cli`.

In [ ]:
!python --version
!pip install -r requirements.txt

## 3. Revisar estructura documental

El Ticket 1 espera una estructura por areas dentro de `documents/` y un inventario maestro en `documents/inventory/BSF-INV-001_Document_Inventory.csv`.

In [ ]:
!find documents -maxdepth 2 -type f | sort

## 4. Cargar el inventario maestro

Esta revision solo lee el CSV oficial. No modifica el inventario.

In [ ]:
from pathlib import Path
import pandas as pd

inventory_path = Path("documents/inventory/BSF-INV-001_Document_Inventory.csv")
inventory_df = pd.read_csv(inventory_path)

print(f"Filas del inventario: {len(inventory_df)}")
print(f"Columnas: {list(inventory_df.columns)}")
inventory_df.head()

## 5. Resumen por area y formato

Este resumen sirve para comprobar que el inventario conserva la lista final de 24 documentos.

In [ ]:
display(inventory_df["business_area"].value_counts().rename_axis("area").reset_index(name="cantidad"))
display(inventory_df["document_format"].value_counts().rename_axis("formato").reset_index(name="cantidad"))

## 6. Validar documentos locales

Este comando compara el inventario oficial contra los archivos disponibles localmente en `documents/<area>/`.

Si solo clonaste GitHub, es normal que aparezcan documentos faltantes porque los documentos fuente no se versionan.

In [ ]:
!python -m rag_bsf.cli validate-documents

## 7. Revisar reporte de validacion

El reporte se genera en `data/processed/document_status.csv`.

In [ ]:
status_path = Path("data/processed/document_status.csv")
status_df = pd.read_csv(status_path)

print(f"Filas del reporte: {len(status_df)}")
display(status_df["status"].value_counts().rename_axis("status").reset_index(name="cantidad"))
status_df.head()

## 8. Listar documentos faltantes

Esta vista ayuda a saber que archivos debes colocar localmente antes de cerrar el Ticket 1.

In [ ]:
missing_df = status_df[status_df["status"] == "missing"]

cols = ["document_id", "title", "category", "expected_folder", "expected_file"]
print(f"Documentos faltantes: {len(missing_df)}")
display(missing_df[cols])

## 9. Generar inventario tecnico

Este comando genera `data/processed/inventory.json` usando el inventario oficial y los Markdown disponibles localmente. No cambia el CSV original.

In [ ]:
!python -m rag_bsf.cli inventory

## 10. Revisar inventario tecnico generado

`inventory.json` muestra que documentos Markdown estan disponibles para una futura ingesta. En Ticket 1 esto es solo evidencia de organizacion.

In [ ]:
import json

technical_inventory_path = Path("data/processed/inventory.json")
technical_inventory = json.loads(technical_inventory_path.read_text(encoding="utf-8"))

print(f"Registros en inventory.json: {len(technical_inventory)}")
technical_df = pd.DataFrame(technical_inventory)
display(technical_df[["document_id", "title", "category", "markdown_available", "local_markdown_path"]].head(10))

## 11. Opcion si quieres validar documentos privados en Colab

Como GitHub no versiona documentos fuente, puedes subir un ZIP privado con tus carpetas `documents/<area>/` y descomprimirlo encima de la estructura del proyecto.

Usa esta celda solo si necesitas comprobar en Colab que los archivos finales existen.

In [ ]:
# Opcional: subir un ZIP local con documentos privados.
# El ZIP debe contener una carpeta documents/ con las subcarpetas corporate, hr, hse,
# operations, quality, it e inventory.

# from google.colab import files
# uploaded = files.upload()
# zip_name = next(iter(uploaded))
# !unzip -o "$zip_name" -d .
# !python -m rag_bsf.cli validate-documents

## 12. Criterio de cierre del Ticket 1

El Ticket 1 queda listo cuando:

- el inventario conserva los 24 documentos esperados;
- las carpetas por area existen;
- `validate-documents` genera `document_status.csv`;
- los documentos finales estan disponibles localmente en sus carpetas;
- los documentos fuente siguen fuera de GitHub;
- `inventory` genera `inventory.json` sin modificar el CSV oficial.

El siguiente paso despues de este cuaderno es el Ticket 2: limpieza de Markdown y generacion de chunks.